# Deceptive Opinions

In [1]:
import keras
import string

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

import gensim.downloader

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Dense, Embedding, Bidirectional, LSTM, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

In [2]:
from google.colab import drive
drive.mount("/content/drive")

file = "/content/drive/MyDrive/deceptive-opinion.csv"
opinions_df = pd.read_csv(file)
opinions_df.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(1600, 5)

In [3]:
opinions_df.head(10)

,deceptive,hotel,polarity,source,text
0,truthful,conrad,positive,TripAdvisor,We stayed for a one night getaway with family ...
1,truthful,hyatt,positive,TripAdvisor,Triple A rate with upgrade to view room was le...
2,truthful,hyatt,positive,TripAdvisor,This comes a little late as I'm finally catchi...
3,truthful,omni,positive,TripAdvisor,The Omni Chicago really delivers on all fronts...
4,truthful,hyatt,positive,TripAdvisor,I asked for a high floor away from the elevato...
5,truthful,omni,positive,TripAdvisor,I stayed at the Omni for one night following a...
6,truthful,conrad,positive,TripAdvisor,We stayed in the Conrad for 4 nights just befo...
7,truthful,omni,positive,TripAdvisor,Just got back from 2 days up in Chicago shoppi...
8,truthful,omni,positive,TripAdvisor,We arrived at the Omni on 2nd September for a ...
9,truthful,hyatt,positive,TripAdvisor,"On our visit to Chicago, we chose the Hyatt du..."


## Dataset Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    opinions_df["text"],
    opinions_df["polarity"],
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

## Feature Engineering

In [5]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_seq = pad_sequences(
    X_train_seq,
    maxlen=200,
    padding="post",
    truncating="post",
)
X_test_seq = pad_sequences(
    X_test_seq,
    maxlen=200,
    padding="post",
    truncating="post",
)

X_train_seq.shape, X_test_seq.shape

((1280, 200), (320, 200))

In [6]:
y_train_array = np.array(y_train)
y_train_array = np.where(y_train_array == "negative", 0, 1)

y_test_array = np.array(y_test)
y_test_array = np.where(y_test_array == "negative", 0, 1)


y_train_seq = tf.one_hot(y_train_array, depth=1)
y_test_seq = tf.one_hot(y_test_array, depth=1)

y_train_seq.shape, y_test_seq.shape

(TensorShape([1280, 1]), TensorShape([320, 1]))

## Word2Vec

In [7]:
news_vectors = gensim.downloader.load("fasttext-wiki-news-subwords-300")

In [8]:
embedded_dim = news_vectors.vector_size
vocab_size = len(tokenizer.word_index) + 1
matrix = np.zeros((vocab_size, embedded_dim))
for word in tokenizer.word_index:
    if word in news_vectors:
        matrix[tokenizer.word_index[word]] = news_vectors[word]

## Modelo

In [9]:
opinions_classifier = keras.Sequential(
    [
        Embedding(
            input_dim=vocab_size,
            output_dim=embedded_dim,
            weights=[matrix],
            trainable=False,
        ),
        Bidirectional(LSTM(256, return_sequences=True)),
        Bidirectional(LSTM(128)),
        Dense(128, activation="relu"),
        Dropout(0.2),
        Dense(1, activation="sigmoid"),
    ]
)

opinions_classifier.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy", "recall", "precision"],
)

opinions_classifier.build(input_shape=(None, 300))

opinions_classifier.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 300)       │     2,791,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 300, 512)       │     1,140,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       656,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,621,345 (17.63 MB)

 Trainable params: 1,830,145 (6.98 MB)

 Non-trainable params: 2,791,200 (10.65 MB)

In [10]:
# Define the callback
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='best_model.keras',
    monitor='val_accuracy',     # Metric to monitor (e.g., 'val_accuracy', 'val_loss')
    mode='max',             # 'min' for loss, 'max' for accuracy
    save_best_only=True,    # Only save when the monitored metric improves
    verbose=1
)


opinions_classifier.fit(
    X_train_seq, y_train_seq, epochs=30, batch_size=32, validation_split=0.2, callbacks=[checkpoint_callback]
)

Epoch 1/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5769 - loss: 0.6774 - precision: 0.5802 - recall: 0.7068
Epoch 1: val_accuracy improved from None to 0.61328, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 9s 84ms/step - accuracy: 0.5996 - loss: 0.6764 - precision: 0.6190 - recall: 0.5500 - val_accuracy: 0.6133 - val_loss: 0.6568 - val_precision: 0.6818 - val_recall: 0.4580
Epoch 2/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.6483 - loss: 0.6352 - precision: 0.6936 - recall: 0.6047
Epoch 2: val_accuracy did not improve from 0.61328
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - accuracy: 0.6582 - loss: 0.6396 - precision: 0.6959 - recall: 0.5808 - val_accuracy: 0.6133 - val_loss: 0.6535 - val_precision: 0.6212 - val_recall: 0.6260
Epoch 3/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6851 - loss: 0.6087 - precision: 0.6937 - recall: 0.7336
Epoch 3: val_accuracy did not improve from 0.6132

In [11]:
opinions_classifier.evaluate(X_test_seq, y_test_seq, batch_size=32)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9344 - loss: 0.1619 - precision: 0.9211 - recall: 0.9396


[0.16185054183006287,
 0.934374988079071,
 0.9395973086357117,
 0.9210526347160339]